# prepare dataset

In [ ]:
# ! pip install googletrans==4.0.0-rc1
# ! pip install sentence-transformers
# ! pip install underthesea
# ! pip install nltk
# ! pip install gensim

In [ ]:
import pandas as pd
import numpy as np
import joblib

In [ ]:
df = pd.read_csv('jobs2k_cleaned.csv')

In [ ]:
# df_recs = pd.read_csv('df_processed.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6473 entries, 0 to 6472
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   job_url             6473 non-null   object 
 1   title               6473 non-null   object 
 2   salary_range        6473 non-null   object 
 3   min_salary          4033 non-null   float64
 4   max_salary          4033 non-null   float64
 5   avg_salary          4033 non-null   float64
 6   location            6473 non-null   object 
 7   region              5725 non-null   object 
 8   description         6473 non-null   object 
 9   requirements        6473 non-null   object 
 10  benefit             6473 non-null   object 
 11  company_url         6473 non-null   object 
 12  company_name        6473 non-null   object 
 13  company_avatar      6473 non-null   object 
 14  company_scale       6473 non-null   object 
 15  company_address     6473 non-null   object 
 16  positi

In [ ]:
df['specializations'] = df['specializations'].str.split(',').apply(lambda x: [i.strip() for i in x])


In [ ]:
df_mini = df[['title','description','requirements','benefit','type','location','position','salary_range','specializations']].copy()

In [ ]:
import re
import numpy as np

def parse_salary_range(salary_string):
    """
    Parses a salary range string and extracts min and max salary as raw numbers.

    Args:
        salary_string (str): The input salary string (e.g., "2,000 - 3,500 USD",
                             "Tới 2,000 USD", "Thoả thuận", "12 - 15 triệu").

    Returns:
        tuple: A tuple containing (min_salary, max_salary).
               Returns (np.nan, np.nan) if no valid numbers or "thoả thuận".
    """
    if not isinstance(salary_string, str):
        return np.nan, np.nan

    salary_string_lower = salary_string.lower()

    if "thoả thuận" in salary_string_lower or "thu nhập, thoả thuận" in salary_string_lower:
        return np.nan, np.nan

    # Extract all numbers, allowing for commas and decimal points
    numbers_str = re.findall(r'[\d.,]+', salary_string_lower)

    numbers = []
    for num_str in numbers_str:
        cleaned_num_str = num_str.replace(',', '')
        try:
            numbers.append(float(cleaned_num_str))
        except ValueError:
            pass # Ignore if conversion fails

    min_val, max_val = np.nan, np.nan

    if not numbers:
        return np.nan, np.nan

    # No unit conversion, just use the raw numbers

    if len(numbers) == 1:
        val = numbers[0]
        if "tới" in salary_string_lower or "lên đến" in salary_string_lower:
            min_val = np.nan
            max_val = val
        elif "từ" in salary_string_lower:
            min_val = val
            max_val = np.nan
        else: # Single number without 'tới' or 'từ', assume exact
            min_val = val
            max_val = val
    elif len(numbers) >= 2:
        min_val = numbers[0]
        max_val = numbers[1]
        # Ensure min_val <= max_val
        if min_val > max_val:
            min_val, max_val = max_val, min_val

    return min_val, max_val
def parse_salary_currency(salary_string):
    """
    Parses a salary string to extract the currency.

    Args:
        salary_string (str): The input salary string (e.g., "2,000 - 3,500 USD",
                             "Tới 2,000 VND", "Thoả thuận").

    Returns:
        str: The extracted currency (e.g., "USD", "VND").
             Returns np.nan if no valid currency is found or "thoả thuận".
    """
    if not isinstance(salary_string, str):
        return np.nan

    salary_string_lower = salary_string.lower()

    if "thoả thuận" in salary_string_lower or "thu nhập, thoả thuận" in salary_string_lower:
        return np.nan

    # Look for common currency codes
    currency_patterns = {
        # 'USD': r'\b(usd|dollar|đô la)\b',
        # 'VND': r'\b(vnd|đồng|vnđ)\b',
        # 'EUR': r'\b(eur|euro)\b',
        # 'JPY': r'\b(jpy|yen|yên)\b',
        # 'GBP': r'\b(gbp|pound|bảng)\b'
        'USD': r'\b(usd)\b',
        'triệu VND': r'\b(triệu)\b'
    }

    for currency, pattern in currency_patterns.items():
        if re.search(pattern, salary_string_lower):
            return currency

    return np.nan

In [ ]:
df_mini['min_salary_edited'], df_mini['max_salary_edited'] = zip(*df_mini['salary_range'].apply(parse_salary_range))

In [ ]:
df_mini['currency_salary'] = df_mini['salary_range'].apply(parse_salary_currency)

In [ ]:
file_path = 'vietnamese-stopwords-dash.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    vietnamese_stopwords = f.read().splitlines()

In [ ]:
import re
import string
from underthesea import word_tokenize as word_tokenize_underthesea

def preprocess_underthesea(text):
    if not isinstance(text, str):
        return ""
    # 1. lowercase
    text = text.lower()

    # 2. remove punctuation
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # 3. tokenization (tách từ)
    tokens = word_tokenize_underthesea(text, format="text")
    token_list = tokens.split()

    # 4. remove stopwords
    filtered = [w for w in token_list if w not in vietnamese_stopwords]

    return " ".join(filtered)

In [ ]:
df_mini.columns

Index(['title', 'description', 'requirements', 'benefit', 'type', 'location',
       'position', 'salary_range', 'specializations', 'min_salary_edited',
       'max_salary_edited', 'currency_salary'],
      dtype='object')

In [ ]:
df_recs = df_mini.copy()

In [ ]:
df_recs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6473 entries, 0 to 6472
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              6473 non-null   object 
 1   description        6473 non-null   object 
 2   requirements       6473 non-null   object 
 3   benefit            6473 non-null   object 
 4   type               6473 non-null   object 
 5   location           6473 non-null   object 
 6   position           6473 non-null   object 
 7   salary_range       6473 non-null   object 
 8   specializations    6473 non-null   object 
 9   min_salary_edited  4299 non-null   float64
 10  max_salary_edited  4299 non-null   float64
 11  currency_salary    4504 non-null   object 
dtypes: float64(2), object(10)
memory usage: 607.0+ KB


In [ ]:
df_recs['min_salary_edited'].fillna(0, inplace=True)
df_recs['max_salary_edited'].fillna(0, inplace=True)
df_recs['currency_salary'].fillna("Unknown", inplace=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_6752\3178225061.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_recs['min_salary_edited'].fillna(0, inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_6752\3178225061.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [ ]:
def combine_text_fields(row):
    fields = ['title', 'description', 'requirements', 'benefit']
    combined_text = '. '.join(str(row[field]) for field in fields if pd.notnull(row[field]))
    return combined_text

In [ ]:
df_recs['overall_text'] = df_recs.apply(combine_text_fields, axis=1)

In [ ]:
df_recs['title_processed'] = df_recs['title'].apply(preprocess_underthesea)
df_recs['overall_text_processed'] = df_recs['overall_text'].apply(preprocess_underthesea)

In [ ]:
df_recs.to_csv('df_processed.csv', index=False)

In [ ]:
# from googletrans import Translator
# import time

# translator = Translator()

# def translate_text(text, dest_lang='en'):
#     if not isinstance(text, str) or not text.strip():
#         return ""

#     # Googletrans has character limits. Split text into chunks if it's too long.
#     # A rough estimate for character limit is around 5000 characters per request.
#     # For simplicity, let's use a conservative chunk size.
#     max_chunk_size = 2000 # Adjust as needed

#     if len(text) > max_chunk_size:
#         chunks = [text[i:i + max_chunk_size] for i in range(0, len(text), max_chunk_size)]
#         translated_chunks = []
#         for chunk in chunks:
#             try:
#                 translated_chunk = translator.translate(chunk, dest=dest_lang).text
#                 translated_chunks.append(translated_chunk)
#                 time.sleep(0.1) # Be respectful to the API by adding a small delay
#             except Exception as e:
#                 print(f"Error translating chunk: {e}")
#                 translated_chunks.append(chunk) # Return original chunk on error
#         return " ".join(translated_chunks)
#     else:
#         try:
#             return translator.translate(text, dest=dest_lang).text
#         except Exception as e:
#             print(f"Error translating text: {e}")
#             return text # Return original text on error

# print("Translator initialized and translate_text function defined.")

Translator initialized and translate_text function defined.


In [ ]:
# import nltk
# nltk.download('punkt_tab')
# nltk.download('stopwords')

In [ ]:
# import re
# import string
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize as word_tokenize_nltk

# def preprocess_english(text):
#     if not isinstance(text, str):
#         return ""

#     # 1. lowercase
#     text = text.lower()

#     # 2. remove punctuation
#     text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()

#     # 3. tokenization (tách từ)
#     token_list = word_tokenize_nltk(text)

#     # 4. remove stopwords
#     filtered = [w for w in token_list if w not in stopwords.words('english')]

#     return " ".join(filtered)

# print("English preprocessing function defined.")

In [ ]:
# df_recs['title_en_processed'] = df_recs['title_en'].apply(preprocess_english)
# df_recs['overall_text_en_processed'] = df_recs['overall_text_en'].apply(preprocess_english)
# # print(df_recs[['title', 'title_en', 'title_en_processed']].head())

In [ ]:
df_recs = pd.read_csv('df_processed.csv')

# TF-IDF nguyên mẫu

## basic (only title)

In [ ]:
input_keyword = 'kỹ sư phần mềm'
preprocessed_keyword = preprocess_underthesea(input_keyword)
print(f"Original keyword: {input_keyword}")
print(f"Preprocessed keyword: {preprocessed_keyword}")

Original keyword: kỹ sư phần mềm
Preprocessed keyword: kỹ_sư phần_mềm


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the 'title_underthesea' column
tfidf_vectorizer.fit(df_recs['title_processed'])

tfidf_matrix = tfidf_vectorizer.transform(df_recs['title_processed'])

joblib.dump(tfidf_vectorizer, "tfidf_model_vi_basic.joblib")

# Optionally, convert the TF-IDF matrix to a DataFrame for better readability
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# print("TF-IDF matrix shape:", tfidf_matrix.shape)
# print("First 5 rows of TF-IDF DataFrame (if created):\n", tfidf_df.head())

In [ ]:
vectorized_keyword = tfidf_vectorizer.transform([preprocessed_keyword])
print(f"Shape of vectorized keyword: {vectorized_keyword.shape}")

Shape of vectorized keyword: (1, 6711)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between the vectorized keyword and all job titles
cosine_similarities = cosine_similarity(vectorized_keyword, tfidf_matrix)

print(f"Shape of cosine similarities: {cosine_similarities.shape}")

Shape of cosine similarities: (1, 6473)


In [ ]:
num_recommendations = 10
similarity_scores = cosine_similarities.flatten()
df_recs['similarity_score'] = similarity_scores

df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

top_jobs = df_filtered.sort_values('similarity_score', ascending=False).head(10)

print(f"Top {num_recommendations} job recommendations for '{input_keyword}':")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']} (score: {row['similarity_score']:.4f})")

Top 10 job recommendations for 'kỹ sư phần mềm':
1. Tuyển Nhân Viên Telesales Phần Mềm làm việc tại Công ty TNHH phát triển AROMA (score: 0.3643)
2. Tuyển Nhân Viên Chăm Sóc Khách Hàng Mảng Phần Mềm làm việc tại Công ty TNHH Pancake Việt Nam (score: 0.3319)
3. Tuyển Chăm Sóc Khách Hàng làm việc tại CÔNG TY CỔ PHẦN PHẦN MỀM CHUYÊN NGHIỆP TOÀN CẦU (score: 0.3307)
4. Tuyển Chuyên Viên Phát Triển Phần Mềm làm việc tại Công ty cổ phần phát triển Hi Tech Việt Nam (score: 0.3196)
5. Tuyển Chuyên Viên Kiểm Thử Phần Mềm /Tester làm việc tại Công ty Cổ phần SIMBATECH (score: 0.3109)
6. Tuyển Chuyên Viên Digital Marketing Sản Phẩm Phần Mềm làm việc tại Công ty Cổ phần Bvote Việt Nam (score: 0.3107)
7. Tuyển Android Developer / Kỹ Sư Phần Mềm Android _ Thu Nhập Upto 30tr làm việc tại CÔNG TY CỔ PHẦN CÔNG NGHỆ VÀ TRUYỀN THÔNG BES (score: 0.3067)
8. Tuyển Manual Tester làm việc tại Công ty cổ phần phần mềm ITSOL Holdings (score: 0.2998)
9. Tuyển Chuyên Viên Kinh Doanh Phần Mềm AI/ Sales B2B làm việc

## upgrade (full label)

In [ ]:
text_keyword = 'kỹ sư phần mềm, nghỉ hai ngày trong tuần, có lương thưởng'
preprocessed_keyword = preprocess_underthesea(text_keyword)
print(f"Original keyword: {text_keyword}")
print(f"Preprocessed keyword: {preprocessed_keyword}")

Original keyword: kỹ sư phần mềm, nghỉ hai ngày trong tuần, có lương thưởng
Preprocessed keyword: kỹ_sư phần_mềm nghỉ hai tuần lương thưởng


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the 'title_underthesea' column
tfidf_vectorizer.fit(df_recs['overall_text_processed'])

tfidf_matrix = tfidf_vectorizer.transform(df_recs['overall_text_processed'])

joblib.dump(tfidf_vectorizer, "tfidf_model_vi_upgrade.joblib")

# Optionally, convert the TF-IDF matrix to a DataFrame for better readability
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# print("TF-IDF matrix shape:", tfidf_matrix.shape)
# print("First 5 rows of TF-IDF DataFrame (if created):\n", tfidf_df.head())

In [ ]:
vectorized_keyword = tfidf_vectorizer.transform([preprocessed_keyword])
print(f"Shape of vectorized keyword: {vectorized_keyword.shape}")

Shape of vectorized keyword: (1, 34766)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between the vectorized keyword and all job titles
cosine_similarities = cosine_similarity(vectorized_keyword, tfidf_matrix)

print(f"Shape of cosine similarities: {cosine_similarities.shape}")

Shape of cosine similarities: (1, 6473)


In [ ]:
num_recommendations = 10
similarity_scores = cosine_similarities.flatten()
df_recs['similarity_score'] = similarity_scores

df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

top_jobs = df_filtered.sort_values('similarity_score', ascending=False).head(10)

print(f"Top {num_recommendations} job recommendations for '{text_keyword}':")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']} (score: {row['similarity_score']:.4f})")

Top 10 job recommendations for 'kỹ sư phần mềm, nghỉ hai ngày trong tuần, có lương thưởng':
1. Tuyển Trợ Lý Kế Toán - Không Yêu Cầu Kinh Nghiệm làm việc tại Công ty TNHH Smart Outsourcing (score: 0.1812)
2. Tuyển Giáo Viên Toán Tư Duy Cấp TH - THCS (Times City _ Hai Bà Trưng/ Nguyễn Sơn _ Long Biên _Hà Nội/TP.HCM) làm việc tại Mathnasium Việt Nam (score: 0.1754)
3. Tuyển Kỹ Sư Triển Khai Bản Vẽ Thi Công (Shopdrawing) làm việc tại CÔNG TY CỔ PHẦN XÂY DỰNG CDC (score: 0.1679)
4. Tuyển Kỹ Sư Cấp Thoát Nước / Kỹ Thuật Hiện Trường / Kỹ Sư Hiện Trường ( Không Yêu Cầu Kinh Nghiệm ) làm việc tại CÔNG TY CỔ PHẦN ĐẦU TƯ XÂY DỰNG KHÔI LÂM (score: 0.1641)
5. Tuyển Kỹ Sư Cơ Điện M&E (ĐIỆN, HVAC, CTN, PCCC...) làm việc tại Công ty Cổ Phần Kỹ thuật Thăng Tiến (score: 0.1506)
6. Tuyển Kỹ Sư Kết Cấu Thép làm việc tại Công ty Cổ phần Thương mại, Tư vấn và Xây dựng Vĩnh Hưng (score: 0.1485)
7. Tuyển Kỹ Sư  Cấp  Thoát Nước làm việc tại Công ty TNHH Đầu tư Phát Triển công nghệ Suntech VN (score: 0.1480)
8.

# Phương pháp lai với các mô hình nhúng đa ngôn ngữ và tương đồng cosine

## BGE M3

### basic (only title)

Data Vietnamese ***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model BGE M3
print("Loading BGE-M3 multilingual SentenceTransformer model...")
model = SentenceTransformer('BAAI/bge-m3')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
input_keyword_embedding = model.encode(
    [input_keyword],
    normalize_embeddings=True
)
print(f"Shape of input keyword embedding: {input_keyword_embedding.shape}")

# 3. Encode job titles
print("Encoding job title embeddings with BGE-M3 (this may take a moment)...")
job_title_embeddings = model.encode(
    df_recs['title_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job title embeddings: {job_title_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    input_keyword_embedding,
    job_title_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (BGE-M3):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

Loading BGE-M3 multilingual SentenceTransformer model...
Model loaded.
Shape of input keyword embedding: (1, 1024)
Encoding job title embeddings with BGE-M3 (this may take a moment)...
Shape of job title embeddings: (6473, 1024)
Shape of cosine similarities: (1, 6473)

Top 10 Job Title Recommendations for 'kỹ sư phần mềm' (BGE-M3):
1. Tuyển Nhân Viên Kỹ Thuật Phần Mềm làm việc tại CÔNG TY TNHH THƯƠNG MẠI TỔNG HỢP HTV (CellphoneS) (Similarity Score: 0.6585)
2. Tuyển Kỹ Sư An Toàn Thông Tin (Security Engineer) làm việc tại Viettel Software (Similarity Score: 0.6357)
3. Tuyển Fresher Automotive làm việc tại FPT Software (Similarity Score: 0.6205)
4. Tuyển Sales Engineer (Bán Hàng Kỹ Thuật) làm việc tại CÔNG TY CỔ PHẦN PMAC (Similarity Score: 0.6033)
5. Tuyển Kỹ Sư Triển Khai Bản Vẽ Kết Cấu (Office) làm việc tại CÔNG TY TNHH KAS E&C (VIỆT NAM) (Similarity Score: 0.6022)
6. Tuyển Nhân Viên Kỹ Thuật Thiết Bị Bộ Đàm Và Máy In làm việc tại Công ty Cổ phần Công nghệ DSS Việt Nam (Similarity Sco

In [ ]:
model.save('bge_m3_model_vn_basic')
np.save('job_title_embeddings_bge_m3_vn_basic.npy', job_title_embeddings)

### upgrade (full label)

Data Vietnamese ***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model BGE M3
print("Loading BGE-M3 multilingual SentenceTransformer model...")
model = SentenceTransformer('BAAI/bge-m3')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
text_keyword_embedding = model.encode(
    [text_keyword],
    normalize_embeddings=True
)
print(f"Shape of text keyword embedding: {text_keyword_embedding.shape}")

# 3. Encode job text
print("Encoding job text embeddings with BGE-M3 (this may take a moment)...")
job_text_embeddings = model.encode(
    df_recs['overall_text_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job text embeddings: {job_text_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    text_keyword_embedding,
    job_text_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (BGE-M3):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

Loading BGE-M3 multilingual SentenceTransformer model...
Model loaded.
Shape of text keyword embedding: (1, 1024)
Encoding job text embeddings with BGE-M3 (this may take a moment)...


In [ ]:
model.save('bge_m3_model_vn_upgrade')
np.save('job_text_embeddings_bge_m3_vn_upgrade.npy', job_text_embeddings)

Data Translate to English

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model BGE M3
print("Loading BGE-M3 multilingual SentenceTransformer model...")
# model = SentenceTransformer('BAAI/bge-m3')
model = SentenceTransformer('BAAI/BGE-base-en')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
translated_text_keyword_en = translate_text(text_keyword, dest_lang='en')
text_keyword_embedding = model.encode(
    [translated_text_keyword_en],
    normalize_embeddings=True
)
print(f"Shape of text keyword embedding: {text_keyword_embedding.shape}")

# 3. Encode job titles (English processed)
print("Encoding job text embeddings with BGE-M3 (this may take a moment)...")
job_text_embeddings = model.encode(
    df_recs['overall_text_en_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job text embeddings: {job_text_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    text_keyword_embedding,
    job_text_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi
print("Similarity scores assigned to DataFrame.")

df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)
# 6. Print results
num_recommendations = 10
top_jobs = df_filtered.sort_values('similarity_score', ascending=False).head(10)

print(f"Top {num_recommendations} job recommendations for '{text_keyword}':")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']} (score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('bge_base_en_model_en_upgrade')
np.save('job_text_embeddings_en_upgrade.npy', job_text_embeddings)

## paraphrase-multilingual-mpnet-base-v2

### basic (only title)

Data Vietnamese ***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model
print("Loading paraphrase-multilingual-mpnet-base-v2 model...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("Model loaded.")

# 2. Encode input keyword
print("Encoding input keyword...")
input_keyword_embedding = model.encode(
    [input_keyword],
    normalize_embeddings=True
)
print("Shape of input keyword embedding:", input_keyword_embedding.shape)

# 3. Encode job titles
print("Encoding job titles...")
job_title_embeddings = model.encode(
    df_recs["title_processed"].tolist(),
    normalize_embeddings=True
)
print("Shape of job title embeddings:", job_title_embeddings.shape)

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    input_keyword_embedding,
    job_title_embeddings
)
print("Shape of cosine similarities:", cosine_similarities_multi.shape)

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (multilingual-mpnet):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('paraphrase_multilingual_mpnet_model_vn_basic')
np.save('job_title_embeddings_mpnet_vn_basic.npy', job_title_embeddings)

Data Translate to English

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model
print("Loading paraphrase-multilingual-mpnet-base-v2 model...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
translated_input_keyword_en = translate_text(input_keyword, dest_lang='en')
input_keyword_embedding = model.encode(
    [translated_input_keyword_en],
    normalize_embeddings=True
)
print(f"Shape of input keyword embedding: {input_keyword_embedding.shape}")

# 3. Encode job titles (English processed)
print("Encoding job title embeddings with BGE-M3 (this may take a moment)...")
job_title_embeddings = model.encode(
    df_recs['title_en_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job title embeddings: {job_title_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    input_keyword_embedding,
    job_title_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (multilingual-mpnet):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('paraphrase_multilingual_mpnet_model_en_basic')
np.save('job_title_embeddings_mpnet_en_basic.npy', job_title_embeddings)

### upgrade (full label)

Data Vietnamese ***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model
print("Loading paraphrase-multilingual-mpnet-base-v2 model...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
text_keyword_embedding = model.encode(
    [text_keyword],
    normalize_embeddings=True
)
print(f"Shape of text keyword embedding: {text_keyword_embedding.shape}")

# 3. Encode job text
print("Encoding job text embeddings with BGE-M3 (this may take a moment)...")
job_text_embeddings = model.encode(
    df_recs['overall_text_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job text embeddings: {job_text_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    text_keyword_embedding,
    job_text_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (multilingual-mpnet):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('paraphrase_multilingual_mpnet_model_vn_upgrade')
np.save('job_text_embeddings_mpnet_vn_upgrade.npy', job_text_embeddings)

Data Translate to English

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model
print("Loading paraphrase-multilingual-mpnet-base-v2 model...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
translated_text_keyword_en = translate_text(text_keyword, dest_lang='en')
text_keyword_embedding = model.encode(
    [translated_text_keyword_en],
    normalize_embeddings=True
)
print(f"Shape of text keyword embedding: {text_keyword_embedding.shape}")

# 3. Encode job titles (English processed)
print("Encoding job text embeddings with BGE-M3 (this may take a moment)...")
job_text_embeddings = model.encode(
    df_recs['overall_text_en_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job text embeddings: {job_text_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    text_keyword_embedding,
    job_text_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (multilingual-mpnet):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('paraphrase_multilingual_mpnet_model_en_upgrade')
np.save('job_text_embeddings_mpnet_en_upgrade.npy', job_text_embeddings)

## LaBSE

### basic (only title)

Data Vietnamese ***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model LaBSE
print("Loading LaBSE multilingual SentenceTransformer model...")
model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
input_keyword_embedding = model.encode(
    [input_keyword],
    normalize_embeddings=True
)
print(f"Shape of input keyword embedding: {input_keyword_embedding.shape}")

# 3. Encode job titles
print("Encoding job title embeddings with LaBSE (this may take a moment)...")
job_title_embeddings = model.encode(
    df_recs['title_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job title embeddings: {job_title_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    input_keyword_embedding,
    job_title_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (LaBSE):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('labse_model_vn_basic')
np.save('job_title_embeddings_labse_vn_basic.npy', job_title_embeddings)

Data Translate to English

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model LaBSE
print("Loading LaBSE multilingual SentenceTransformer model...")
model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
translated_input_keyword_en = translate_text(input_keyword, dest_lang='en')
input_keyword_embedding = model.encode(
    [translated_input_keyword_en],
    normalize_embeddings=True
)
print(f"Shape of input keyword embedding: {input_keyword_embedding.shape}")

# 3. Encode job titles (English processed)
print("Encoding job title embeddings with LaBSE (this may take a moment)...")
job_title_embeddings = model.encode(
    df_recs['title_en_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job title embeddings: {job_title_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    input_keyword_embedding,
    job_title_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (LaBSE):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('labse_model_en_basic')
np.save('job_title_embeddings_labse_en_basic.npy', job_title_embeddings)

### upgrade (full label)

Data Vietnamese ***

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model LaBSE
print("Loading LaBSE multilingual SentenceTransformer model...")
model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
text_keyword_embedding = model.encode(
    [text_keyword],
    normalize_embeddings=True
)
print(f"Shape of text keyword embedding: {text_keyword_embedding.shape}")

# 3. Encode job text
print("Encoding job text embeddings with LaBSE (this may take a moment)...")
job_text_embeddings = model.encode(
    df_recs['overall_text_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job text embeddings: {job_text_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    text_keyword_embedding,
    job_text_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (LaBSE):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

In [ ]:
model.save('labse_model_vn_upgrade')
np.save('job_text_embeddings_labse_vn_upgrade.npy', job_text_embeddings)

Data Translate to English

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load model LaBSE
print("Loading LaBSE multilingual SentenceTransformer model...")
model = SentenceTransformer('sentence-transformers/LaBSE')
print("Model loaded.")

# 2. Encode the original Vietnamese keyword
translated_text_keyword_en = translate_text(text_keyword, dest_lang='en')
text_keyword_embedding = model.encode(
    [translated_text_keyword_en],
    normalize_embeddings=True
)
print(f"Shape of text keyword embedding: {text_keyword_embedding.shape}")

# 3. Encode job titles (English processed)
print("Encoding job text embeddings with LaBSE (this may take a moment)...")
job_text_embeddings = model.encode(
    df_recs['overall_text_en_processed'].tolist(),
    normalize_embeddings=True
)
print(f"Shape of job text embeddings: {job_text_embeddings.shape}")

# 4. Cosine similarity
cosine_similarities_multi = cosine_similarity(
    text_keyword_embedding,
    job_text_embeddings
)
print(f"Shape of cosine similarities: {cosine_similarities_multi.shape}")

# 5. Identify top matches
similarity_scores_multi = cosine_similarities_multi.flatten()
df_recs["similarity_score_multi"] = similarity_scores_multi

# 6. Print results
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary
)
num_recommendations = 10
top_jobs_multi = df_filtered.sort_values("similarity_score_multi", ascending=False).head(num_recommendations)

# 8. In kết quả
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (LaBSE):")
for i, (_, row) in enumerate(top_jobs_multi.iterrows(), start=1):
    print(f"{i}. {row['title']} (Similarity Score: {row['similarity_score_multi']:.4f})")

# Word2Vec

## Tính trung bình cộng (Average) tất cả vector của các từ trong văn bản.

In [ ]:
tokenized_sentences = [text.split() for text in df_recs['title_processed']]
print("Tokenized sentences prepared for Word2Vec training.")
print(tokenized_sentences[:5])

Tokenized sentences prepared for Word2Vec training.
[['tuyển', 'kế_toán', 'tổng_hợp', 'đi', 'thu_nhập', 'upto', '20', 'triệu', '3', 'kinh_nghiệm', 'ktth', 'làm_việc', 'công_ty', 'cổ_phần', 'bền'], ['tuyển', 'business', 'development_executive', 'làm_việc', 'công_ty', 'cổ_phần', 'golden', 'owl', 'solutions'], ['tuyển', 'nhân_viên', 'content', 'community_marketing', 'làm_việc', 'công_ty', 'cổ_phần', 'heligate'], ['tuyển', 'trưởng', 'kinh_doanh', 'làm_việc', 'công_ty', 'tnhh', 'mtv', 'k', 'tech'], ['tuyển', 'nhân_viên', 'kinh_doanh', 'giải_pháp', 'cntt', 'làm_việc', 'công_ty', 'cổ_phần', 'tư_vấn', 'đào_tạo', 'smartpro']]


In [ ]:
tokenized_sentences_text = [text.split() for text in df_recs['overall_text_processed']]
print("Tokenized sentences prepared for Word2Vec training.")
print(tokenized_sentences_text[:5])

Tokenized sentences prepared for Word2Vec training.
[['tuyển', 'kế_toán', 'tổng_hợp', 'đi', 'thu_nhập', 'upto', '20', 'triệu', '3', 'kinh_nghiệm', 'ktth', 'làm_việc', 'công_ty', 'cổ_phần', 'bền_·', 'kiểm_tra', 'kiểm_soát', 'chứng_từ', 'kế_toán', '·_lập', 'quy_định', '·', 'hạch_toán', 'tổng_hợp', 'nghiệp_vụ', 'kế_toán', '·_lập', 'định_kỳ', 'quý', '·_lập', 'kế_hoạch', 'giám_sát', '·', 'tncn', 'gtgt', 'tndn', 'nhà_thầu', '…', '·', 'phối_hợp', 'kế_toán_trưởng', 'làm_việc', '·', 'giám_sát', 'đào_tạo', 'nhân_viên', 'kế_toán', '·', 'công_việc', 'nam_nữ', 'ưu_tiên', 'ứng_viên', '32', 'tốt_nghiệp', 'đại_học', 'chuyên_ngành', 'tối_thiểu', '3', 'ưu_tiên', 'lĩnh_vực', 'thương_mại', 'thành_thạo', 'phần_mềm', 'kế_toán', 'am_hiểu', 'kỹ_năng', 'phân_tích', 'tổ_chức', 'công_việc', 'tư_duy', 'logic', 'hệ_thống', '·', 'môi_trường', 'làm_việc', '·', 'năng_lực', 'dựa', '·', 'bhxh', 'bhyt', 'nghỉ', 'lễ_tết', 'phép', 'lương', '13', '·', 'tham_gia', 'ứng_dụng'], ['tuyển', 'business', 'development_executive', 

In [ ]:
import numpy as np

def get_document_embedding(text_tokens, w2v_model):
    embeddings = []
    for word in text_tokens:
        if word in w2v_model.wv:
            embeddings.append(w2v_model.wv[word])

    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        # Return a zero vector if no words are found in the vocabulary
        return np.zeros(w2v_model.vector_size)

print("get_document_embedding function defined.")

get_document_embedding function defined.


### sg=0 là CBOW (mặc định)

#### basic (only title)

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec model
print("Training Word2Vec model...")
w2v_model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, workers=4)
print("Word2Vec model trained successfully.")
print(f"Number of words in vocabulary: {len(w2v_model.wv)}")

Training Word2Vec model...
Word2Vec model trained successfully.
Number of words in vocabulary: 6751


In [ ]:
print("Creating document embeddings for all job titles...")
df_recs['title_w2v_embedding'] = df_recs['title_processed'].apply(lambda x: get_document_embedding(x.split(), w2v_model))
print("Document embeddings created successfully.")
print(f"Shape of the first document embedding: {df_recs['title_w2v_embedding'].iloc[0].shape}")
print(df_recs[['title', 'title_w2v_embedding']].head())

Creating document embeddings for all job titles...
Document embeddings created successfully.
Shape of the first document embedding: (100,)
                                               title  \
0  Tuyển Kế Toán Tổng Hợp Đi Làm Ngay Thu Nhập Up...   
1  Tuyển Business Development Executive làm việc ...   
2  Tuyển Nhân Viên Content Community Marketing là...   
3  Tuyển Trưởng Nhóm Kinh Doanh làm việc tại CÔNG...   
4  Tuyển Nhân Viên Kinh Doanh Giải Pháp CNTT làm ...   

                                 title_w2v_embedding  
0  [-0.05079646, 0.5626101, 0.20427234, -0.079379...  
1  [-0.07813389, 0.44473886, 0.0045683514, -0.003...  
2  [-0.12254049, 0.5685925, 0.0027146374, 0.01871...  
3  [-0.1346162, 0.60280424, 0.011522859, 0.031264...  
4  [-0.14617325, 0.6084221, 0.0037530966, 0.02821...  


In [ ]:
# input_keyword = "Chuyên viên phân tích dữ liệu Data Analyst biết SQL"

In [ ]:
processed_input_keyword_w2v = preprocess_underthesea(input_keyword)
print(f"Preprocessed input keyword for Word2Vec: {processed_input_keyword_w2v}")

tokenized_input_keyword_w2v = processed_input_keyword_w2v.split()
input_keyword_embedding_w2v = get_document_embedding(tokenized_input_keyword_w2v, w2v_model)

print(f"Shape of vectorized input keyword for Word2Vec: {input_keyword_embedding_w2v.shape}")

Preprocessed input keyword for Word2Vec: chuyên_viên phân_tích dữ_liệu data analyst sql
Shape of vectorized input keyword for Word2Vec: (100,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape the input keyword embedding to be a 2D array for cosine_similarity
input_keyword_embedding_w2v_reshaped = input_keyword_embedding_w2v.reshape(1, -1)

# Convert the list of job title embeddings to a 2D numpy array
job_title_embeddings_w2v = np.array(df_recs['title_w2v_embedding'].tolist())

# Calculate cosine similarity
cosine_similarities_w2v = cosine_similarity(input_keyword_embedding_w2v_reshaped, job_title_embeddings_w2v)
print(f"Shape of cosine similarities for Word2Vec: {cosine_similarities_w2v.shape}")

Shape of cosine similarities for Word2Vec: (1, 6473)


In [ ]:
similarity_scores_w2v = cosine_similarities_w2v.flatten()
df_recs['similarity_score_w2v'] = similarity_scores_w2v
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)
# df_filtered = df_recs.copy()

# 8. Sort + top N
num_recommendations = 10
top_jobs = df_filtered.sort_values(
    "similarity_score_w2v",
    ascending=False
).head(num_recommendations)

# 9. Print results
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (Word2Vec):")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']}  (Similarity: {row['similarity_score_w2v']:.4f})")


Top 10 Job Title Recommendations for 'Chuyên viên phân tích dữ liệu Data Analyst biết SQL' (Word2Vec):
1. Tuyển Nhân Viên Kinh Doanh/ Chuyên Viên Tư Vấn Hoạch Định Tài Chính - Không Yêu Cầu Kinh Nghiệm Bảo Hiểm làm việc tại ASAHI LIFE CONSULTING VIETNAM CO.,LTD  (Similarity: 0.9993)
2. Tuyển Nhân Viên Kinh Doanh/ Chuyên Viên Tư Vấn Hoạch Định Tài Chính - Không Yêu Cầu Kinh Nghiệm Bảo Hiểm làm việc tại ASAHI LIFE CONSULTING VIETNAM CO.,LTD  (Similarity: 0.9993)
3. Tuyển Nhân Viên Marketing Online (Content & Media – Facebook & TikTok) làm việc tại HỘ KINH DOANH KY MI  (Similarity: 0.9993)
4. Tuyển Nhân Viên Kinh Doanh Part-Time (Biết Tiếng Trung) làm việc tại Hộ Kinh Doanh Kim Phong Food  (Similarity: 0.9993)
5. Tuyển Nhân Viên Thiết Kế 3D – Full Time làm việc tại HỘ KINH DOANH PHẠM THANH HẢI 3  (Similarity: 0.9993)
6. Tuyển Nhân Viên Sale Online (Trực Page) làm việc tại HỘ KINH DOANH THE FAMA  (Similarity: 0.9992)
7. Tuyển Nhân Viên Pha Chế Và Phục Vụ Fulltime làm việc tại HỘ KINH DOAN

In [ ]:
w2v_model.wv.save("word2vec_average_job_titles.kv")
np.save('job_title_embeddings_w2v_ave_vn_basic.npy', job_title_embeddings_w2v)

#### upgrade (full label)

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec model
print("Training Word2Vec model...")
w2v_model = Word2Vec(sentences=tokenized_sentences_text, vector_size=100, window=5, min_count=1, workers=4)
print("Word2Vec model trained successfully.")
print(f"Number of words in vocabulary: {len(w2v_model.wv)}")

Training Word2Vec model...
Word2Vec model trained successfully.
Number of words in vocabulary: 3889


In [ ]:
print("Creating document embeddings for all job overralls...")
df_recs['overall_text_w2v_embedding'] = df_recs['overall_text_processed'].apply(lambda x: get_document_embedding(x.split(), w2v_model))
print("Document embeddings created successfully.")
print(f"Shape of the first document embedding: {df_recs['overall_text_w2v_embedding'].iloc[0].shape}")
print(df_recs[['overall_text', 'overall_text_w2v_embedding']].head())

Creating document embeddings for all job overralls...
Document embeddings created successfully.
Shape of the first document embedding: (100,)
                                        overall_text  \
0  Tuyển Kỹ Thuật Viên Sửa Chữa Bảo Dưỡng Ô Tô (T...   
1  Tuyển Huấn Luyện Viên Cá Nhân (PT) - Không Yêu...   
2  Tuyển Kỹ Sư Giám Sát Cơ Điện/MEP/ Giám Sát Thi...   
3  Tuyển Chuyên Viên Kỹ Thuật Hiện Trường làm việ...   
4  Tuyển Nhân Viên Kiểm Soát Nội Bộ làm việc tại ...   

                          overall_text_w2v_embedding  
0  [-0.010770225, 0.021579342, 0.0026548074, 0.00...  
1  [-0.01340779, 0.023480106, 0.0029089577, 0.001...  
2  [-0.012525476, 0.024370128, 0.0035626832, 0.00...  
3  [-0.01167516, 0.023760982, 0.0033952498, 0.000...  
4  [-0.010094878, 0.020624552, 0.0026296205, 0.00...  


In [ ]:
processed_text_keyword_w2v = preprocess_underthesea(text_keyword)
print(f"Preprocessed text keyword for Word2Vec: {processed_text_keyword_w2v}")

tokenized_text_keyword_w2v = processed_text_keyword_w2v.split()
text_keyword_embedding_w2v = get_document_embedding(tokenized_text_keyword_w2v, w2v_model)

print(f"Shape of vectorized text keyword for Word2Vec: {text_keyword_embedding_w2v.shape}")

Preprocessed text keyword for Word2Vec: kỹ_sư phần_mềm nghỉ hai tuần lương thưởng
Shape of vectorized text keyword for Word2Vec: (100,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape the text keyword embedding to be a 2D array for cosine_similarity
text_keyword_embedding_w2v_reshaped = text_keyword_embedding_w2v.reshape(1, -1)

# Convert the list of job overall embeddings to a 2D numpy array
job_overall_embeddings_w2v = np.array(df_recs['title_w2v_embedding'].tolist())

# Calculate cosine similarity
cosine_similarities_w2v = cosine_similarity(text_keyword_embedding_w2v_reshaped, job_overall_embeddings_w2v)
print(f"Shape of cosine similarities for Word2Vec: {cosine_similarities_w2v.shape}")

Shape of cosine similarities for Word2Vec: (1, 19)


In [ ]:
similarity_scores_w2v = cosine_similarities_w2v.flatten()
df_recs['similarity_score_w2v'] = similarity_scores_w2v
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

# 8. Sort + top N
num_recommendations = 10
top_jobs = df_filtered.sort_values(
    "similarity_score_w2v",
    ascending=False
).head(num_recommendations)

# 9. Print results
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (Word2Vec):")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']}  (Similarity: {row['similarity_score_w2v']:.4f})")

In [ ]:
w2v_model.wv.save("word2vec_average_job_overall.kv")
np.save('job_overall_embeddings_w2v_ave_vn_upgrade.npy', job_overall_embeddings_w2v)

### sg=1 là Skip-gram

#### basic (only title)

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec model
print("Training Word2Vec model...")
w2v_model_sg = Word2Vec(sg=1, sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, workers=4)
print("Word2Vec model trained successfully.")
print(f"Number of words in vocabulary: {len(w2v_model_sg.wv)}")

Training Word2Vec model...
Word2Vec model trained successfully.
Number of words in vocabulary: 486


In [ ]:
print("Creating document embeddings for all job titles...")
df_recs['title_w2v_embedding_sg'] = df_recs['title_processed'].apply(lambda x: get_document_embedding(x.split(), w2v_model_sg))
print("Document embeddings created successfully.")
print(f"Shape of the first document embedding: {df_recs['title_w2v_embedding_sg'].iloc[0].shape}")
print(df_recs[['title', 'title_w2v_embedding_sg']].head())

Creating document embeddings for all job titles...
Document embeddings created successfully.
Shape of the first document embedding: (100,)
                                               title  \
0  Tuyển Quản Lý Sản Xuất / Kiểm Soát Chất Lượng ...   
1  Tuyển Trưởng Nhóm Thiết Kế Design (Design Team...   
2  Tuyển IT Project Manager (Regulatory Banking P...   
3  Tuyển Nhân Viên Tư Vấn Bán Hàng làm việc tại C...   
4  Tuyển Giám Sát Thi Công/ Công Trình Xây Dựng (...   

                              title_w2v_embedding_sg  
0  [-0.007979045, 0.0107745435, 0.00083289464, -0...  
1  [-0.005892513, 0.009658699, -0.0016723316, -0....  
2  [-0.005960188, 0.008526837, 0.00090207387, -0....  
3  [-0.007595903, 0.011631543, 0.0007456219, 0.00...  
4  [-0.006483274, 0.011126444, 0.00074656657, -0....  


In [ ]:
processed_input_keyword_w2v_sg = preprocess_underthesea(input_keyword)
print(f"Preprocessed input keyword for Word2Vec: {processed_input_keyword_w2v_sg}")

tokenized_input_keyword_w2v_sg = processed_input_keyword_w2v.split()
input_keyword_embedding_w2v_sg = get_document_embedding(tokenized_input_keyword_w2v_sg, w2v_model_sg)

print(f"Shape of vectorized input keyword for Word2Vec: {input_keyword_embedding_w2v_sg.shape}")

Preprocessed input keyword for Word2Vec: kỹ_sư phần_mềm
Shape of vectorized input keyword for Word2Vec: (100,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape the input keyword embedding to be a 2D array for cosine_similarity
input_keyword_embedding_w2v_sg_reshaped = input_keyword_embedding_w2v_sg.reshape(1, -1)

# Convert the list of job title embeddings to a 2D numpy array
job_title_embeddings_w2v_sg = np.array(df_recs['title_w2v_embedding_sg'].tolist())

# Calculate cosine similarity
cosine_similarities_w2v_sg = cosine_similarity(input_keyword_embedding_w2v_sg_reshaped, job_title_embeddings_w2v_sg)
print(f"Shape of cosine similarities for Word2Vec: {cosine_similarities_w2v_sg.shape}")

Shape of cosine similarities for Word2Vec: (1, 98)


In [ ]:
similarity_scores_w2v_sg = cosine_similarities_w2v_sg.flatten()
df_recs['similarity_score_w2v'] = similarity_scores_w2v_sg
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

# 8. Sort + top N
num_recommendations = 10
top_jobs = df_filtered.sort_values(
    "similarity_score_w2v",
    ascending=False
).head(num_recommendations)

# 9. Print results
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (Word2Vec):")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']}  (Similarity: {row['similarity_score_w2v']:.4f})")

#### upgrade (full label)

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec model
print("Training Word2Vec model...")
w2v_model_sg = Word2Vec(sentences=tokenized_sentences_text, vector_size=100, window=5, min_count=1, workers=4)
print("Word2Vec model trained successfully.")
print(f"Number of words in vocabulary: {len(w2v_model_sg.wv)}")

Training Word2Vec model...
Word2Vec model trained successfully.
Number of words in vocabulary: 3889


In [ ]:
print("Creating document embeddings for all job overralls...")
df_recs['overall_text_w2v_embedding'] = df_recs['overall_text_processed'].apply(lambda x: get_document_embedding(x.split(), w2v_model_sg))
print("Document embeddings created successfully.")
print(f"Shape of the first document embedding: {df_recs['overall_text_w2v_embedding'].iloc[0].shape}")
print(df_recs[['overall_text', 'overall_text_w2v_embedding']].head())

Creating document embeddings for all job overralls...
Document embeddings created successfully.
Shape of the first document embedding: (100,)
                                        overall_text  \
0  Tuyển Kỹ Thuật Viên Sửa Chữa Bảo Dưỡng Ô Tô (T...   
1  Tuyển Huấn Luyện Viên Cá Nhân (PT) - Không Yêu...   
2  Tuyển Kỹ Sư Giám Sát Cơ Điện/MEP/ Giám Sát Thi...   
3  Tuyển Chuyên Viên Kỹ Thuật Hiện Trường làm việ...   
4  Tuyển Nhân Viên Kiểm Soát Nội Bộ làm việc tại ...   

                          overall_text_w2v_embedding  
0  [-0.009950863, 0.021377383, 0.002825029, 0.001...  
1  [-0.012539086, 0.023247011, 0.003099481, 0.000...  
2  [-0.011607613, 0.024134029, 0.0037516083, 0.00...  
3  [-0.010790723, 0.023534408, 0.003584976, 0.000...  
4  [-0.009304225, 0.02042183, 0.0028021673, 0.000...  


In [ ]:
processed_text_keyword_w2v_sg = preprocess_underthesea(text_keyword)
print(f"Preprocessed text keyword for Word2Vec: {processed_text_keyword_w2v_sg}")

tokenized_text_keyword_w2v_sg = processed_text_keyword_w2v_sg.split()
text_keyword_embedding_w2v_sg = get_document_embedding(tokenized_text_keyword_w2v_sg, w2v_model_sg)

print(f"Shape of vectorized text keyword for Word2Vec: {text_keyword_embedding_w2v_sg.shape}")

Preprocessed text keyword for Word2Vec: kỹ_sư phần_mềm nghỉ hai tuần lương thưởng
Shape of vectorized text keyword for Word2Vec: (100,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape the text keyword embedding to be a 2D array for cosine_similarity
text_keyword_embedding_w2v_reshaped_sg = text_keyword_embedding_w2v_sg.reshape(1, -1)

# Convert the list of job overall embeddings to a 2D numpy array
job_overall_embeddings_w2v_sg = np.array(df_recs['title_w2v_embedding'].tolist())

# Calculate cosine similarity
cosine_similarities_w2v_sg = cosine_similarity(text_keyword_embedding_w2v_reshaped_sg, job_overall_embeddings_w2v_sg)
print(f"Shape of cosine similarities for Word2Vec: {cosine_similarities_w2v_sg.shape}")

Shape of cosine similarities for Word2Vec: (1, 19)


In [ ]:
similarity_scores_w2v_sg = cosine_similarities_w2v_sg.flatten()
df_recs['similarity_score_w2v'] = similarity_scores_w2v_sg
df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

# 8. Sort + top N
num_recommendations = 10
top_jobs = df_filtered.sort_values(
    "similarity_score_w2v",
    ascending=False
).head(num_recommendations)

# 9. Print results
print(f"\nTop {num_recommendations} Job Title Recommendations for '{text_keyword}' (Word2Vec):")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']}  (Similarity: {row['similarity_score_w2v']:.4f})")

## Dùng Doc2Vec.

### basic (only title)

In [ ]:
from gensim.models.doc2vec import TaggedDocument

# Create a list of TaggedDocument objects
print("Preparing data in TaggedDocument format for Doc2Vec training...")
tagged_data = [TaggedDocument(words=text.split(), tags=[str(i)]) for i, text in enumerate(df_recs['title_processed'])]

print("Data prepared in TaggedDocument format.")
print(f"Total number of tagged documents: {len(tagged_data)}")
print("First 2 tagged documents:")
print(tagged_data[:2])

Preparing data in TaggedDocument format for Doc2Vec training...
Data prepared in TaggedDocument format.
Total number of tagged documents: 98
First 2 tagged documents:
[TaggedDocument(words=['tuyển', 'quản_lý', 'sản_xuất', 'kiểm_soát', 'chất_lượng', 'qc', 'thu_nhập', '12', '16', 'triệu', 'khu_vực', 'long_an', 'cũ', 'làm_việc', 'công_ty', 'tnhh', 'mtv', 'trung_sơn', 'long_an'], tags=['0']), TaggedDocument(words=['tuyển', 'trưởng', 'thiết_kế', 'design', 'design', 'team', 'leader', 'ngành', 'nhà_hàng', 'khách_sạn', 'thu_nhập', '30', 'triệu', '1', 'làm_việc', 'công_ty', 'cổ_phần', 'quản_lý', 'khách_sạn', 'odyssea'], tags=['1'])]


In [ ]:
from gensim.models.doc2vec import Doc2Vec

# Initialize Doc2Vec model
print("Initializing Doc2Vec model...")
doc2vec_model = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=20)

# Build vocabulary
print("Building Doc2Vec vocabulary...")
doc2vec_model.build_vocab(tagged_data)

# Train the model
print("Training Doc2Vec model...")
doc2vec_model.train(tagged_data, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
print("Doc2Vec model trained successfully.")

Initializing Doc2Vec model...
Building Doc2Vec vocabulary...
Training Doc2Vec model...
Doc2Vec model trained successfully.


In [ ]:
print("Inferring document embeddings for job titles...")
doc_embeddings = []
for doc in tagged_data:
    doc_embeddings.append(doc2vec_model.infer_vector(doc.words))

df_recs['title_doc2vec_embedding'] = pd.Series(doc_embeddings)

print("Document embeddings inferred and added to df_recs.")
print(f"Shape of the first document embedding: {df_recs['title_doc2vec_embedding'].iloc[0].shape}")
print(df_recs[['title', 'title_doc2vec_embedding']].head())

Inferring document embeddings for job titles...
Document embeddings inferred and added to df_recs.
Shape of the first document embedding: (100,)
                                               title  \
0  Tuyển Quản Lý Sản Xuất / Kiểm Soát Chất Lượng ...   
1  Tuyển Trưởng Nhóm Thiết Kế Design (Design Team...   
2  Tuyển IT Project Manager (Regulatory Banking P...   
3  Tuyển Nhân Viên Tư Vấn Bán Hàng làm việc tại C...   
4  Tuyển Giám Sát Thi Công/ Công Trình Xây Dựng (...   

                             title_doc2vec_embedding  
0  [-0.058467004, 0.03261783, -0.05040646, -0.060...  
1  [-0.0755755, 0.041620813, -0.060063194, -0.073...  
2  [-0.048010413, 0.030426562, -0.039645746, -0.0...  
3  [-0.024167525, 0.010850736, -0.022484707, -0.0...  
4  [-0.065976046, 0.03699273, -0.04459019, -0.065...  


In [ ]:
processed_input_keyword_doc2vec = preprocess_underthesea(input_keyword)
print(f"Preprocessed input keyword for Doc2Vec: {processed_input_keyword_doc2vec}")

tokenized_input_keyword_doc2vec = processed_input_keyword_doc2vec.split()
input_keyword_embedding_doc2vec = doc2vec_model.infer_vector(tokenized_input_keyword_doc2vec)

print(f"Shape of vectorized input keyword for Doc2Vec: {input_keyword_embedding_doc2vec.shape}")

Preprocessed input keyword for Doc2Vec: kỹ_sư phần_mềm
Shape of vectorized input keyword for Doc2Vec: (100,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape the input keyword embedding to be a 2D array for cosine_similarity
input_keyword_embedding_doc2vec_reshaped = input_keyword_embedding_doc2vec.reshape(1, -1)

# Convert the list of job title embeddings to a 2D numpy array
job_title_embeddings_doc2vec = np.array(df_recs['title_doc2vec_embedding'].tolist())

# Calculate cosine similarity
cosine_similarities_doc2vec = cosine_similarity(input_keyword_embedding_doc2vec_reshaped, job_title_embeddings_doc2vec)
print(f"Shape of cosine similarities for Doc2Vec: {cosine_similarities_doc2vec.shape}")

Shape of cosine similarities for Doc2Vec: (1, 98)


In [ ]:
similarity_scores_doc2vec = cosine_similarities_doc2vec.flatten()
df_recs['similarity_score_doc2vec'] = similarity_scores_doc2vec

df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

# 8. Sort results and return top-k
num_recommendations = 10
top_jobs = df_filtered.sort_values(
    "similarity_score_doc2vec",
    ascending=False
).head(num_recommendations)

# 9. Print results
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (Doc2Vec):")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']}  (Score: {row['similarity_score_doc2vec']:.4f})")

In [ ]:
doc2vec_model.save("word2vec_doc2vec_job_title.model")
np.save('job_title_embeddings_doc2vec_vn_basic.npy', job_title_embeddings_doc2vec)

### upgrade (full label)

In [ ]:
from gensim.models.doc2vec import TaggedDocument

# Create a list of TaggedDocument objects
print("Preparing data in TaggedDocument format for Doc2Vec training...")
tagged_data = [TaggedDocument(words=text.split(), tags=[str(i)]) for i, text in enumerate(df_recs['overall_text_processed'])]

print("Data prepared in TaggedDocument format.")
print(f"Total number of tagged documents: {len(tagged_data)}")
print("First 2 tagged documents:")
print(tagged_data[:2])

Preparing data in TaggedDocument format for Doc2Vec training...
Data prepared in TaggedDocument format.
Total number of tagged documents: 19
First 2 tagged documents:
[TaggedDocument(words=['tuyển', 'kỹ_thuật_viên', 'sửa_chữa', 'bảo_dưỡng', 'ô_tô', 'thu_nhập', '15', '25', 'triệu', 'hà_nội', 'hồ', 'chí_minh', 'làm_việc', 'công_ty', 'tnhh', 'dịch_vụ', 'vận_tải', 'sinh_thái', 'vinbus', 'tchăm', 'sức', 'khỏe', 'chế_độ', 'phúc_lợi', 'hấp_dẫn', 'đóng', 'bảo_hiểm', 'quy_định', 'team', 'building', 'du_lịch', 'hàng', 'chế_độ', 'đãi_ngộ', 'hấp_dẫn', 'dịch_vụ', 'tập_đoàn', 'vingroup', 'bảo_hiểm', 'chăm_sóc', 'sức', 'khỏe', 'phụ_cấp', 'ăn_ca', 'khám', 'sức', 'khỏe_định_kỳ', 'lương', 'thưởng', 'dịp', 'lễ_tết', 'môi_trường', 'làm_việc', 'công_bằng', 'văn_minh', 'lộ_trình', 'thăng_tiến', 'rõ_ràng'], tags=['0']), TaggedDocument(words=['tuyển', 'huấn_luyện_viên', 'pt', 'kinh_nghiệm', 'toàn_quốc', 'làm_việc', 'california', 'fitness', 'yoga', 'centers', 'tư_vấn', 'khách_hàng', 'sức_khỏe', 'chế_độ', 'dinh

In [ ]:
from gensim.models.doc2vec import Doc2Vec

# Initialize Doc2Vec model
print("Initializing Doc2Vec model...")
doc2vec_model = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=20)

# Build vocabulary
print("Building Doc2Vec vocabulary...")
doc2vec_model.build_vocab(tagged_data)

# Train the model
print("Training Doc2Vec model...")
doc2vec_model.train(tagged_data, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
print("Doc2Vec model trained successfully.")

Initializing Doc2Vec model...
Building Doc2Vec vocabulary...
Training Doc2Vec model...
Doc2Vec model trained successfully.


In [ ]:
print("Inferring document embeddings for job overalls...")
doc_embeddings = []
for doc in tagged_data:
    doc_embeddings.append(doc2vec_model.infer_vector(doc.words))

df_recs['overall_text_doc2vec_embedding'] = pd.Series(doc_embeddings)

print("Document embeddings inferred and added to df_recs.")
print(f"Shape of the first document embedding: {df_recs['overall_text_doc2vec_embedding'].iloc[0].shape}")
print(df_recs[['overall_text', 'overall_text_doc2vec_embedding']].head())

Inferring document embeddings for job overalls...
Document embeddings inferred and added to df_filtered.
Shape of the first document embedding: (100,)
                                        overall_text  \
0  Tuyển Kỹ Thuật Viên Sửa Chữa Bảo Dưỡng Ô Tô (T...   
1  Tuyển Huấn Luyện Viên Cá Nhân (PT) - Không Yêu...   
2  Tuyển Kỹ Sư Giám Sát Cơ Điện/MEP/ Giám Sát Thi...   
3  Tuyển Chuyên Viên Kỹ Thuật Hiện Trường làm việ...   
4  Tuyển Nhân Viên Kiểm Soát Nội Bộ làm việc tại ...   

                      overall_text_doc2vec_embedding  
0  [-0.81045586, -0.39877358, -0.034876734, -0.33...  
1  [-1.1570476, -0.57265013, -0.047474064, -0.503...  
2  [-1.0921818, -0.5355763, -0.035615407, -0.4592...  
3  [-1.2619628, -0.6187888, -0.043337725, -0.5339...  
4  [-1.323766, -0.6668674, -0.04858559, -0.573456...  


In [ ]:
processed_text_keyword_doc2vec = preprocess_underthesea(text_keyword)
print(f"Preprocessed text keyword for Doc2Vec: {processed_text_keyword_doc2vec}")

tokenized_text_keyword_doc2vec = processed_text_keyword_doc2vec.split()
text_keyword_embedding_doc2vec = doc2vec_model.infer_vector(tokenized_text_keyword_doc2vec)

print(f"Shape of vectorized text keyword for Doc2Vec: {text_keyword_embedding_doc2vec.shape}")

Preprocessed text keyword for Doc2Vec: kỹ_sư phần_mềm nghỉ hai tuần lương thưởng
Shape of vectorized text keyword for Doc2Vec: (100,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Reshape the text keyword embedding to be a 2D array for cosine_similarity
text_keyword_embedding_doc2vec_reshaped = text_keyword_embedding_doc2vec.reshape(1, -1)

# Convert the list of job overall embeddings to a 2D numpy array
job_overall_embeddings_doc2vec = np.array(df_recs['overall_text_doc2vec_embedding'].tolist())

# Calculate cosine similarity
cosine_similarities_doc2vec = cosine_similarity(text_keyword_embedding_doc2vec_reshaped, job_overall_embeddings_doc2vec)
print(f"Shape of cosine similarities for Doc2Vec: {cosine_similarities_doc2vec.shape}")

Shape of cosine similarities for Doc2Vec: (1, 19)


In [ ]:
similarity_scores_doc2vec = cosine_similarities_doc2vec.flatten()
df_recs['similarity_score_doc2vec'] = similarity_scores_doc2vec

df_filtered = filter_sample(df_recs, re_type, re_position, re_location, re_salary_min, re_currency_salary)

# 8. Sort results and return top-k
num_recommendations = 10
top_jobs = df_filtered.sort_values(
    "similarity_score_doc2vec",
    ascending=False
).head(num_recommendations)

# 9. Print results
print(f"\nTop {num_recommendations} Job Title Recommendations for '{input_keyword}' (Doc2Vec):")
for i, (_, row) in enumerate(top_jobs.iterrows(), start=1):
    print(f"{i}. {row['title']}  (Score: {row['similarity_score_doc2vec']:.4f})")

In [ ]:
doc2vec_model.save("word2vec_doc2vec_job_overall.model")
np.save('job_overall_embeddings_doc2vec_vn_upgrade.npy', job_overall_embeddings_doc2vec)

## read pdf

In [ ]:
import pdfplumber

path = "CV_Võ Phi Thân.pdf"

text = ""
with pdfplumber.open(path) as pdf:
    for page in pdf.pages:
        text += page.extract_text(x_tolerance=1, y_tolerance=1) + "\n"

print(text)


VÕ PHI THÂN
Data Engineer
· · ·
TP. Hồ Chí Minh vothan472@gmail.com 0364884084 Github
TÓM TẮT
Tôi hiện là sinh viên năm ba ngành Công nghệ Thông tin tại Trường Đại học Công nghệ Thông tin – Đại học Quốc gia
TP. HCM. Bên cạnh việc học chính khóa, tôi đam mê tự học về Machine Learning thông qua các khóa học trực tuyến.
Tôi đặc biệt quan tâm đến Khai phá dữ liệu, Xử lý ngôn ngữ tự nhiên (NLP), và Trực quan hóa dữ liệu, với mục tiêu trở
thành Kỹ sư Dữ liệu trong tương lai gần.
HỌC VẤN
Trường Đại học Công nghệ Thông tin – ĐHQG-HCM 2022 – Nay
Cử nhân Công nghệ Thông tin – GPA: 3.3/4.0
• Hướng nghiên cứu: Natural Language Processing (NLP), General Web Crawling, Analytics, Large Language
Model (LLM)
• Tiếng Anh: TOEIC 480
KỸ NĂNG
• Ngôn ngữ lập trình: Python, Java, C++, SQL
• Framework: Apache Spark, Django, Tensorflow
• Hệ điều hành: Windows, Ubuntu, Linux
• Cơ sở dữ liệu: MySQL, SQLite, MongoDB, Microsoft SQL Server
• Nền tảng: Git, Docker
• Khác: Giải quyết vấn đề, Sáng tạo, Làm việc nhóm, 

phương pháp -> list_kn -> áp lọc với các điều kiện type, position, location, min_salary -> đưa ra top 10 phù hợp nhất

phương pháp kn theo
-> tên công việc -> title
-> chi tiết công việc -> description
-> yêu cầu công việc -> requirements
-> quyền lợi công việc -> benefit